# GGUF validation notebook

Run this on Kaggle with T4 and the competition SDK/model datasets attached. It writes the strict validation summary consumed by `make submit-ready`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/AI-Agent-Security')]
    for candidate in candidates:
        if (candidate / 'tools' / 'run_gguf_validation.py').exists():
            return candidate
    raise FileNotFoundError('tools/run_gguf_validation.py not found; run from the repo root')


ROOT = find_repo_root()
os.chdir(ROOT)
print('repo root:', ROOT)
gpu = subprocess.run(['nvidia-smi', '-L'], text=True, capture_output=True)
print(gpu.stdout.strip() or gpu.stderr.strip() or 'no GPU visible')

In [ ]:
os.environ.setdefault('GPT_OSS_GGUF_REPO', 'unsloth/gpt-oss-20b-GGUF')
os.environ.setdefault('GPT_OSS_GGUF_FILE', 'gpt-oss-20b-Q4_K_M.gguf')
os.environ.setdefault('GEMMA_GGUF_REPO', 'unsloth/gemma-4-26B-A4B-it-GGUF')
os.environ.setdefault('GEMMA_GGUF_FILE', 'gemma-4-26B-A4B-it-UD-Q4_K_M.gguf')

# For offline Kaggle runs, set these to attached dataset file paths before running:
# os.environ['GPT_OSS_MODEL_PATH'] = '/kaggle/input/<dataset>/gpt-oss-20b-Q4_K_M.gguf'
# os.environ['GEMMA_MODEL_PATH'] = '/kaggle/input/<dataset>/gemma-4-26B-A4B-it-UD-Q4_K_M.gguf'

for name in ('GPT_OSS_MODEL_PATH', 'GEMMA_MODEL_PATH'):
    if os.getenv(name):
        print(name, os.getenv(name))

In [ ]:
cmd = [
    sys.executable,
    'tools/run_gguf_validation.py',
    '--n', os.getenv('VALIDATION_N', '20'),
    '--models', os.getenv('VALIDATION_MODELS', 'gpt_oss,gemma'),
    '--budget-per-model', os.getenv('VALIDATION_BUDGET_PER_MODEL', '3000'),
    '--max-tool-hops', os.getenv('VALIDATION_MAX_TOOL_HOPS', '8'),
    '--env-selection', os.getenv('VALIDATION_ENV_SELECTION', 'gym'),
    '--out', 'research/results/validation-summary.latest.json',
    '--raw-out', 'research/results/validation-raw.latest.jsonl',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json

summary_path = Path('research/results/validation-summary.latest.json')
summary = json.loads(summary_path.read_text())
print(json.dumps({
    'schema_version': summary.get('schema_version'),
    'validation_n': summary.get('validation_n'),
    'max_tool_hops': summary.get('max_tool_hops'),
    'backend': summary.get('backend'),
    'results': summary.get('results'),
}, indent=2, sort_keys=True))